In [4]:
import numpy as np
import pandas as pd

In [78]:
df_base = pd.read_csv("/work/nvme/bdnb/atekkey/outputs_base/val_results_pf.csv")
df_db = pd.read_csv("/work/nvme/bdnb/atekkey/outputs_db/sav_val/db_val_results_pf.csv")

In [96]:
def cleanDf(df):
    df = df.copy()
    df.columns = df.columns.str.replace(' ', '', regex=False)
    df.drop(columns=['obj'], inplace=True)
    df[['filename', 'objnum']] = df['sequence'].str.split('/', n=1, expand=True)
    df.drop(columns=['sequence'], inplace=True)
    return df

def maximizeDfs(df1, df2):
    key_cols = ['filename', 'objnum', 'fnum']
    df1['key'] = df1[key_cols].astype(str).agg('_'.join, axis=1)
    df2['key'] = df2[key_cols].astype(str).agg('_'.join, axis=1)

    df1 = df1[df1['key'].isin(df2['key'])].copy()
    df2 = df2[df2['key'].isin(df1['key'])].copy()


    df1.sort_values(by=['filename', 'objnum'], inplace=True)
    df2.sort_values(by=['filename', 'objnum'], inplace=True)
    
    df_out = df2.copy()
    for i in range(len(df1)):
        if df1.iloc[i]['J&F'] > df2.iloc[i]['J&F']:
            df_out.iloc[i] = df1.iloc[i]
    return df_out


def splitDf(df):
    df_list = []
    for filename in df['filename'].unique():
        df_list.append(df[df['filename'] == filename])
    return df_list

def sumDfList(df_list):
    out_list = []
    for i, df in enumerate(df_list):
        this_sum = 0
        for obj in df["objnum"].unique():
            sep_df = df[df["objnum"] == obj]
            this_sum += sep_df["J&F"].mean()
        this_sum /= df["objnum"].nunique()
        out_list.append(this_sum)
    return round(float(np.array(out_list).mean()), 2)

In [90]:
df_base_clean = cleanDf(df_base)
df_db_clean = cleanDf(df_db)

In [91]:
df_best = maximizeDfs(df_base_clean, df_db_clean)

In [97]:
df_list = splitDf(df_best)
meann = sumDfList(df_list)
meann

85.56